<a href="https://colab.research.google.com/github/henryparfait/Time-Series-Forecasting/blob/main/notebooks/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01  Data acquisition, memory-efficient processing, and sanity checks



In [ ]:
# ---- 1. Setup and configuration -------------------------------------------------
import os, io, json, time, zipfile, hashlib, platform, gc
from pathlib import Path
from contextlib import contextmanager
import numpy as np
import pandas as pd
import psutil
import requests

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/milan_forecasting')   # all outputs go here
ZIP_DIR     = Path('/content/drive/MyDrive/dataverse_files')      # shortcut to the shared folder
PROC_DIR    = PROJECT_DIR / 'processed'
DAILY_DIR   = PROC_DIR / 'daily'
RAW_TMP     = Path('/content/raw_tmp')                             # local scratch for API downloads
for d in (DAILY_DIR, RAW_TMP):
    d.mkdir(parents=True, exist_ok=True)

DOI  = 'doi:10.7910/DVN/EGZHFV'
BASE = 'https://dataverse.harvard.edu'

DAYS     = pd.date_range('2013-11-01', '2014-01-01', freq='D')          # 62 daily files
EXPECTED = {f'sms-call-internet-mi-{d:%Y-%m-%d}.txt': d for d in DAYS}
COLS     = ['square_id', 'time_ms', 'country', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet']

# Global time grid in LOCAL Milan time (CET = UTC+1 for the whole period; DST ended 27 Oct 2013)
ORIGIN_MS = int(pd.Timestamp('2013-11-01 00:00', tz='Europe/Rome').value // 10**6)
assert ORIGIN_MS == 1383260400000
STEP_MS = 600_000                   # 10 minutes
N_INT   = len(DAYS) * 144           # 8,928 intervals
N_SQ    = 10_000
TIME_INDEX = pd.date_range('2013-11-01 00:00', periods=N_INT, freq='10min', tz='Europe/Rome')

# Hardware record (needed later for the training/execution-time section)
vm = psutil.virtual_memory(); du = psutil.disk_usage('/content')
HARDWARE = {'platform': platform.platform(), 'processor': platform.processor(),
            'logical_cpus': psutil.cpu_count(), 'ram_gb': round(vm.total / 1e9, 1),
            'disk_free_gb': round(du.free / 1e9, 1)}
print(HARDWARE)

Mounted at /content/drive
{'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'processor': 'x86_64', 'logical_cpus': 2, 'ram_gb': 13.6, 'disk_free_gb': 93.8}


In [ ]:
# ---- 2. Inventory: which days are in the Drive zips, which must come from the API ---
def zip_inventory(zip_dir):
    inv, other = {}, []
    if not zip_dir.exists():
        print(f'Zip folder not found: {zip_dir} (will use the API for everything)')
        return inv
    for zp in sorted(zip_dir.glob('*.zip')):
        try:
            with zipfile.ZipFile(zp) as z:
                for info in z.infolist():
                    name = Path(info.filename).name
                    if name in EXPECTED:
                        inv.setdefault(name, (zp, info.filename, info.file_size))   # ignore duplicates
                    elif name:
                        other.append(f'{zp.name}:{info.filename}')
        except zipfile.BadZipFile:
            print(f'WARNING: {zp.name} is not a valid zip (incomplete upload?) - skipped')
    if other:
        print('Other members found in zips (ignored):', other[:10])
    return inv

def api_inventory():
    r = requests.get(f'{BASE}/api/datasets/:persistentId/', params={'persistentId': DOI}, timeout=60)
    r.raise_for_status()
    inv = {}
    for f in r.json()['data']['latestVersion']['files']:
        df = f['dataFile']
        if df.get('filename') in EXPECTED:
            md5 = df.get('md5') or (df.get('checksum') or {}).get('value')
            inv[df['filename']] = {'id': df['id'], 'md5': md5, 'size': df.get('filesize')}
    return inv

zinv = zip_inventory(ZIP_DIR)
missing_from_zips = [n for n in EXPECTED if n not in zinv]
print(f'Days found in Drive zips: {len(zinv)}/62')
print('Missing from zips:', [n[21:31] for n in missing_from_zips])

ainv = {}
if missing_from_zips:
    ainv = api_inventory()
    print(f'Days listed by the Dataverse API: {len(ainv)}/62')
    unrecoverable = [n for n in missing_from_zips if n not in ainv]
    assert not unrecoverable, f'No source for: {unrecoverable}'

Days found in Drive zips: 62/62
Missing from zips: []


In [ ]:
# ---- 3. Core functions: open one day as a stream, process it compactly ------------
@contextmanager
def day_stream(name):
    # Yields a binary file handle for one daily file, from the Drive zip if present,
    # otherwise downloaded from the Dataverse API (MD5-verified, deleted after use).
    if name in zinv:
        zp, member, _ = zinv[name]
        with zipfile.ZipFile(zp) as z, z.open(member) as fh:
            yield fh
        return
    meta, path, h = ainv[name], RAW_TMP / name, hashlib.md5()
    with requests.get(f"{BASE}/api/access/datafile/{meta['id']}", stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, 'wb') as out:
            for chunk in r.iter_content(chunk_size=8 << 20):
                out.write(chunk); h.update(chunk)
    if meta['md5'] and h.hexdigest() != meta['md5']:
        path.unlink(missing_ok=True)
        raise IOError(f'MD5 mismatch for {name}')
    try:
        with open(path, 'rb') as fh:
            yield fh
    finally:
        path.unlink(missing_ok=True)

def read_optimised(fh):
    # Only the 3 needed columns, with compact dtypes.
    return pd.read_csv(fh, sep='\t', header=None, names=COLS,
                       usecols=['square_id', 'time_ms', 'internet'],
                       dtype={'square_id': 'uint16', 'time_ms': 'int64', 'internet': 'float32'})

def aggregate_day(df):
    # Sum internet activity over country codes -> dense block [10,000 squares x width intervals].
    # Blank internet fields mean 'no activity' and are treated as 0.
    sq  = df['square_id'].to_numpy()
    idx = (df['time_ms'].to_numpy() - ORIGIN_MS) // STEP_MS
    assert sq.min() >= 1 and sq.max() <= N_SQ, 'unexpected square id'
    ok = (idx >= 0) & (idx < N_INT)
    n_outside = int((~ok).sum())
    sq, idx = sq[ok].astype(np.int64) - 1, idx[ok]
    val = np.nan_to_num(df['internet'].to_numpy()[ok], nan=0.0).astype(np.float64)
    t0, width = int(idx.min()), int(idx.max() - idx.min() + 1)
    flat = sq * width + (idx - t0)
    block = np.bincount(flat, weights=val, minlength=N_SQ * width).reshape(N_SQ, width)
    rows_per_interval = np.bincount(idx - t0, minlength=width)
    return {'t0': t0, 'block': block.astype(np.float32),      # sum in float64, store float32
            'rows_per_interval': rows_per_interval, 'n_rows': int(len(df)), 'n_outside': n_outside}

## Memory evidence: naive vs optimised loading (one day)
The naive approach loads all 8 columns with pandas defaults (int64/float64/object).
The optimised approach reads only the 3 needed columns with compact dtypes, sums across
country codes, and stores the day as a dense float32 block. The results go into `memory_evidence.json`.

In [ ]:
# ---- 4. Memory evidence on the first day -------------------------------------------
proc = psutil.Process()
def rss_mb(): return proc.memory_info().rss / 1e6
first = next(iter(EXPECTED))
evidence = {'file': first}

gc.collect(); r0 = rss_mb(); t = time.perf_counter()
with day_stream(first) as fh:
    naive = pd.read_csv(fh, sep='\t', header=None, names=COLS)
evidence['naive'] = {'rows': len(naive), 'dtypes': naive.dtypes.astype(str).to_dict(),
                     'df_mb': naive.memory_usage(deep=True).sum() / 1e6,
                     'rss_increase_mb': rss_mb() - r0, 'load_s': time.perf_counter() - t}
del naive; gc.collect()

r0 = rss_mb(); t = time.perf_counter()
with day_stream(first) as fh:
    opt = read_optimised(fh)
evidence['optimised_read'] = {'df_mb': opt.memory_usage(deep=True).sum() / 1e6,
                              'rss_increase_mb': rss_mb() - r0, 'load_s': time.perf_counter() - t}
res = aggregate_day(opt); del opt; gc.collect()
evidence['aggregated_day_block_mb'] = res['block'].nbytes / 1e6
evidence['full_dataset'] = {
    'naive_estimate_gb (62 x naive day)': evidence['naive']['df_mb'] * 62 / 1e3,
    'final_matrix_mb (10,000 x 8,928 float32)': N_SQ * N_INT * 4 / 1e6}
evidence['hardware'] = HARDWARE
(PROJECT_DIR / 'memory_evidence.json').write_text(json.dumps(evidence, indent=2, default=str))
print(json.dumps(evidence, indent=2, default=str))

{
  "file": "sms-call-internet-mi-2013-11-01.txt",
  "naive": {
    "rows": 4842625,
    "dtypes": {
      "square_id": "int64",
      "time_ms": "int64",
      "country": "int64",
      "sms_in": "float64",
      "sms_out": "float64",
      "call_in": "float64",
      "call_out": "float64",
      "internet": "float64"
    },
    "df_mb": 309.928132,
    "rss_increase_mb": 342.331392,
    "load_s": 7.079144645000042
  },
  "optimised_read": {
    "df_mb": 67.796882,
    "rss_increase_mb": 83.85740800000002,
    "load_s": 9.157864463999886
  },
  "aggregated_day_block_mb": 5.76,
  "full_dataset": {
    "naive_estimate_gb (62 x naive day)": 19.215544184,
    "final_matrix_mb (10,000 x 8,928 float32)": 357.12
  },
  "hardware": {
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "processor": "x86_64",
    "logical_cpus": 2,
    "ram_gb": 13.6,
    "disk_free_gb": 93.8
  }
}


## Process all 62 days (resumable)

In [ ]:
# ---- 5. Main loop ------------------------------------------------------------------
log_path = PROC_DIR / 'processing_log.csv'
log = pd.read_csv(log_path).to_dict('records') if log_path.exists() else []
t_all = time.perf_counter()
for name, day in EXPECTED.items():
    out = DAILY_DIR / f'{day:%Y-%m-%d}.npz'
    if out.exists():
        continue
    t = time.perf_counter()
    with day_stream(name) as fh:
        res = aggregate_day(read_optimised(fh))
    np.savez_compressed(out, **{k: np.asarray(v) for k, v in res.items()})
    rec = {'day': f'{day:%Y-%m-%d}', 'source': 'zip' if name in zinv else 'api', 't0': res['t0'],
           'width': res['block'].shape[1], 'n_rows': res['n_rows'], 'n_outside': res['n_outside'],
           'seconds': round(time.perf_counter() - t, 1), 'peak_rss_mb': round(rss_mb())}
    log.append(rec); pd.DataFrame(log).to_csv(log_path, index=False)
    print(rec)
    gc.collect()
print(f'Done in {(time.perf_counter() - t_all) / 60:.1f} min')

{'day': '2013-11-01', 'source': 'zip', 't0': 0, 'width': 144, 'n_rows': 4842625, 'n_outside': 0, 'seconds': 9.4, 'peak_rss_mb': 288}
{'day': '2013-11-02', 'source': 'zip', 't0': 144, 'width': 144, 'n_rows': 4745086, 'n_outside': 0, 'seconds': 8.7, 'peak_rss_mb': 314}
{'day': '2013-11-03', 'source': 'zip', 't0': 288, 'width': 144, 'n_rows': 4722579, 'n_outside': 0, 'seconds': 10.2, 'peak_rss_mb': 277}
{'day': '2013-11-04', 'source': 'zip', 't0': 432, 'width': 144, 'n_rows': 5639488, 'n_outside': 0, 'seconds': 6.4, 'peak_rss_mb': 299}
{'day': '2013-11-05', 'source': 'zip', 't0': 576, 'width': 144, 'n_rows': 5874140, 'n_outside': 0, 'seconds': 9.0, 'peak_rss_mb': 310}
{'day': '2013-11-06', 'source': 'zip', 't0': 720, 'width': 144, 'n_rows': 5912097, 'n_outside': 0, 'seconds': 9.0, 'peak_rss_mb': 306}
{'day': '2013-11-07', 'source': 'zip', 't0': 864, 'width': 144, 'n_rows': 5886883, 'n_outside': 0, 'seconds': 7.7, 'peak_rss_mb': 362}
{'day': '2013-11-08', 'source': 'zip', 't0': 1008, 'widt

In [ ]:
# ---- 6. Assemble the full matrix ---------------------------------------------------
M = np.zeros((N_SQ, N_INT), dtype=np.float32)
rows_per_interval = np.zeros(N_INT, dtype=np.int64)
for day in DAYS:
    z = np.load(DAILY_DIR / f'{day:%Y-%m-%d}.npz')
    t0, block = int(z['t0']), z['block']
    w = min(block.shape[1], N_INT - t0)
    M[:, t0:t0 + w] += block[:, :w]              # additive -> safe if files overlap day boundaries
    rows_per_interval[t0:t0 + w] += z['rows_per_interval'][:w]
np.save(PROC_DIR / 'internet_matrix.npy', M)
np.save(PROC_DIR / 'rows_per_interval.npy', rows_per_interval)
print(f'Matrix {M.shape}, {M.nbytes / 1e6:.0f} MB, dtype {M.dtype}')

Matrix (10000, 8928), 357 MB, dtype float32


## Sanity checks and small exports

In [ ]:
# ---- 7. Sanity checks ---------------------------------------------------------------
log_df = pd.read_csv(log_path)
print('Rows outside the 1 Nov-1 Jan local-time grid:', int(log_df['n_outside'].sum()),
      '(0 if files are split by local day; small counts at the edges if split by UTC day)')
print('First interval index per day (0,144,288,... means files split by local day):')
print(log_df[['day', 't0', 'width']].head(4).to_string(index=False))

empty = TIME_INDEX[rows_per_interval == 0]
print(f'\nIntervals with NO records at all (possible outages): {len(empty)}')
if len(empty):
    s = pd.Series(1, index=empty)
    print(s.groupby(s.index.date).size().to_string())

city = pd.Series(M.sum(axis=0), index=TIME_INDEX, name='city_total')
low = city[city < 0.2 * city.rolling(1008, center=True, min_periods=144).median()]
print(f'\nIntervals where city-wide traffic < 20% of weekly rolling median: {len(low)}')

Rows outside the 1 Nov-1 Jan local-time grid: 0 (0 if files are split by local day; small counts at the edges if split by UTC day)
First interval index per day (0,144,288,... means files split by local day):
       day  t0  width
2013-11-01   0    144
2013-11-02 144    144
2013-11-03 288    144
2013-11-04 432    144

Intervals with NO records at all (possible outages): 0

Intervals where city-wide traffic < 20% of weekly rolling median: 0


In [ ]:
# ---- 8. Totals, top-3 areas, and small exports ---------------------------------------
totals = pd.Series(M.sum(axis=1), index=pd.RangeIndex(1, N_SQ + 1, name='square_id'), name='total_internet')
top3 = totals.nlargest(3)
print('Top-3 squares by total Internet activity:\n', top3)
SELECTED = list(dict.fromkeys([int(i) for i in top3.index] + [4159, 4556]))   # de-duplicated, order kept
json.dump({'top3': [int(i) for i in top3.index], 'selected': [int(i) for i in SELECTED]},
          open(PROC_DIR / 'selected_squares.json', 'w'))

totals.to_csv(PROC_DIR / 'square_totals.csv')
sel = pd.DataFrame({f'sq_{s}': M[s - 1] for s in SELECTED}, index=TIME_INDEX)
sel.index.name = 'time'
sel['city_total'] = city.values
sel['rows_in_interval'] = rows_per_interval
sel.to_csv(PROC_DIR / 'selected_series.csv')
print('\nZero-activity intervals per selected square:\n', (sel[[f"sq_{s}" for s in SELECTED]] == 0).sum())
print('\nSaved to', PROC_DIR)

Top-3 squares by total Internet activity:
 square_id
5161    12740061.0
5059    11170854.0
5259    10485779.0
Name: total_internet, dtype: float32

Zero-activity intervals per selected square:
 sq_5161    0
sq_5059    0
sq_5259    0
sq_4159    0
sq_4556    0
dtype: int64

Saved to /content/drive/MyDrive/milan_forecasting/processed
